In [1]:
!pip install datasets transformers accelerate

In [2]:
import os

In [3]:
from datasets import load_dataset


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.0.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "C:\Users\harsh\anaconda3\lib\runpy.py", line 197, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "C:\Users\harsh\anaconda3\lib\runpy.py", line 87, in _run_code
    exec(code, run_globals)
  File "C:\Users\harsh\anaconda3\lib\site-packages\ipykernel_launcher.py", line 16, in <module>
    app.launch_new_instance()
  File "C:\Users\harsh\anaconda3\lib\site-packages\traitlets\config\application.py", line 846, in launch_instance
    app.start()
  File "C:\Users\harsh\anaconda3\lib\site-pack

AttributeError: _ARRAY_API not found


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.0.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "C:\Users\harsh\anaconda3\lib\runpy.py", line 197, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "C:\Users\harsh\anaconda3\lib\runpy.py", line 87, in _run_code
    exec(code, run_globals)
  File "C:\Users\harsh\anaconda3\lib\site-packages\ipykernel_launcher.py", line 16, in <module>
    app.launch_new_instance()
  File "C:\Users\harsh\anaconda3\lib\site-packages\traitlets\config\application.py", line 846, in launch_instance
    app.start()
  File "C:\Users\harsh\anaconda3\lib\site-pack

AttributeError: _ARRAY_API not found

In [4]:
pip install tf-keras

In [5]:
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling,
    pipeline,
)

In [6]:
def load_and_prepare_dataset(file_path):
    print("Loading dataset...")
    dataset = load_dataset("json", data_files=file_path, split="train")

    def format(example):
        return {
            "text": example["prompt"] + " " + example["response"]
        }

    dataset = dataset.map(format)

    return dataset

In [7]:
def tokenize_dataset(dataset, tokenizer):
    def tokenize(example):
        return tokenizer(
            example["text"],
            truncation=True,
            padding="max_length",
            max_length=512
        )
    print("Tokenizing dataset...")
    return dataset.map(tokenize, batched=True)


In [8]:
def train_model(tokenized_dataset, model, tokenizer, output_dir):
    print("Starting training...")
    training_args = TrainingArguments(
        output_dir=output_dir,
        overwrite_output_dir=True,
        num_train_epochs=3,
        per_device_train_batch_size=4,
        save_steps=500,
        save_total_limit=2,
        logging_steps=100,
        fp16=True if torch.cuda.is_available() else False,
    )

    data_collator = DataCollatorForLanguageModeling(
        tokenizer=tokenizer, mlm=False
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_dataset,
        tokenizer=tokenizer,
        data_collator=data_collator,
    )

    trainer.train()
    print("Training complete.")

    trainer.save_model(output_dir)
    tokenizer.save_pretrained(output_dir)
    print(f"Model saved to {output_dir}")

In [9]:
def run_inference(model_path, question_prompt):
    print("Running inference...")
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model = AutoModelForCausalLM.from_pretrained(model_path)

    qa_pipeline = pipeline("text-generation", model=model, tokenizer=tokenizer)

    result = qa_pipeline(question_prompt, max_new_tokens=100, do_sample=True, temperature=0.7)
    print("\nAnswer:")
    print(result[0]["generated_text"])


In [10]:
if __name__ == "__main__":
    import torch

    # Configurable paths
    dataset_file = "qa_dataset4.jsonl"         # <- your JSONL dataset file
    model_name = "gpt2"                       # Try "gpt2" or switch to a larger model if you have resources
    output_model_dir = "./qa_finetuned_model"

    # Load tokenizer and model
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    
    if tokenizer.pad_token is None:
        print("Setting pad_token...")
        tokenizer.pad_token = tokenizer.eos_token
        
    model = AutoModelForCausalLM.from_pretrained(model_name)

    # Load, format, and tokenize dataset
    dataset = load_and_prepare_dataset(dataset_file)
    tokenized_dataset = tokenize_dataset(dataset, tokenizer)

    # Train and save the model
    train_model(tokenized_dataset, model, tokenizer, output_model_dir)

    # Test the model
    question = "Q: How do I reset my password?\nA:"
    run_inference(output_model_dir, question)

Setting pad_token...
Loading dataset...
Tokenizing dataset...
Starting training...


C:\Users\harsh\AppData\Local\Temp/ipykernel_20884/3682667478.py:18: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
C:\Users\harsh\anaconda3\lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
`loss_type=None` was set in the config but it is unrecognised.Using the default loss: `ForCausalLMLoss`.


Step,Training Loss


Training complete.
Model saved to ./qa_finetuned_model
Running inference...


Device set to use cpu



Answer:
Q: How do I reset my password?
A: To resolve issues related to password reset, please contact our customer support at (701)679-4833x5430x75. You can reach them via 569-764-542x8840. To resolve issues related to password reset, please contact our customer support at (701)679-4833x5430x75. You can reach them via 569-764-542x8840. You can reach them via 569-764-5


In [11]:
question = "Q: please reset password?\nA:"
run_inference(output_model_dir, question)

Running inference...


Device set to use cpu



Answer:
Q: please reset password?
A: If you're facing a problem with password reset, we recommend checking your account settings or calling us. You can reach our about us on 847-839-8878x6468. You can reach our about us on 847-839-8878x6468. Find us on Twitter: @CJW_W.


In [12]:
question = "Q: Explain complaints process?\nA:"
run_inference(output_model_dir, question)

Running inference...


Device set to use cpu



Answer:
Q: Explain complaints process?
A: Our support team can help with complaints. You can reach them via (972)547-4428x8484. You can reach them via (972)545-4828x8610. You can reach them via (972)445-3763. You can reach them via (972)443-4760. You can reach them via (972)972-3944. You can reach them via (972)716-4864


In [13]:
question = "Q: What is your name?\nA:"
run_inference(output_model_dir, question)

Running inference...


Device set to use cpu



Answer:
Q: What is your name?
A: Our support team can help with name management. You can reach them via +1.546.764x4641.66423.5454.9073x741.944.7528.1128.822.1301x816.636.85416.7738.9245.8837


In [14]:
question = "Q: Explain shipping process?\nA:"
run_inference(output_model_dir, question)

Running inference...


Device set to use cpu



Answer:
Q: Explain shipping process?
A: Our support team can help with shipping. You can reach them via +1-336-863-3748x867. Visit their site for more info.


In [15]:
question = "Q: Whom to contact for technical support?\nA:"
run_inference(output_model_dir, question)

Running inference...


Device set to use cpu



Answer:
Q: Whom to contact for technical support?
A: Our support team can help with technical support. You can reach them via +1-253-355-2360x731. You can reach them via +1-252-376-6048x639. You can reach them via +1-252-447-4848x4067. You can reach them via +1-252-447-4848x3363. You can reach them via +1-252-447-4848x3730. You


In [16]:
question = "Q: How to do technical support?\nA:"
run_inference(output_model_dir, question)

Running inference...


Device set to use cpu



Answer:
Q: How to do technical support?
A: Our support team can help with technical support. You can reach them via 004-622-3622x766. You can reach them via 004-622-622-3423. You can reach them via 004-622-3039. You can reach them via 004-622-737. You can reach them via 004-622-737-9283. You can reach them via 004-622-9


In [17]:
question = "Q: Hello,what is the weather today?\nA:"
run_inference(output_model_dir, question)

Running inference...


Device set to use cpu



Answer:
Q: Hello,what is the weather today?
A: Our team can help with weather. You can reach them via 001-541-764-5252. You can reach them via 001-541-764-4851. You can find detailed information about weather on our help center page. Visit our site for more info.


In [18]:
question = "Q: Explain me about billing?\nA:"
run_inference(output_model_dir, question)

Running inference...


Device set to use cpu



Answer:
Q: Explain me about billing?
A: Our support team can help with billing. You can reach them via (647)971-7253. To resolve issues related to billing, please contact them via (647)971-7839. You can reach them via (647)971-8591. You can reach them via (647)971-8630. You can reach them via (647)971-7916. You can reach them via (647)971-5201. You can


In [19]:
question = "Q: Need information about subscribtions?\nA:"
run_inference(output_model_dir, question)

Running inference...


Device set to use cpu



Answer:
Q: Need information about subscribtions?
A: You can find detailed information about subscriptions on our help center page. You can reach us via +1-766-872-7866x5251. You can reach us via +1-766-872-7866x5251. You can reach us via +1-766-872-7866x5251. You can reach us via +1-766-872-7866x5251. You can reach us via +1


In [20]:
question = "Q: Could you explain me about the subscriptions?\nA:"
run_inference(output_model_dir, question)

Running inference...


Device set to use cpu



Answer:
Q: Could you explain me about the subscriptions?
A: Our support team can help with subscriptions. You can reach them via +1-933-497-3255. You can reach them via +1-764-934-1660x637. You can reach them via +1-834-857-9081-6181. You can reach them via +1-543-839-1245. You can reach them via +1-766-873-56423.


In [21]:
question = "Q: What is the capital of India?\nA:"
run_inference(output_model_dir, question)

Running inference...


Device set to use cpu



Answer:
Q: What is the capital of India?
A: India is a country of about 6.7 million people. It is home to a wide variety of industries and services, including finance, telecom, retail, and health. Visit our site for more info.
